<h1 align=center style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
Semantic Segmentation of Geometric Shapes Using UNet
</font>
</h1>

<p dir=ltr style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
In this project, our goal is to train a deep learning model (UNet) for <b>semantic segmentation</b> of images; meaning that the model must determine for each pixel of the image which class it belongs to (background, circle, rectangle, triangle). This project has been implemented step-by-step with simple and scientific explanations to make it easily understandable for beginners.
</font>
</p>

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from tqdm import trange
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from tensorflow.keras.metrics import MeanIoU
import cv2

<div dir="ltr">

<h4 dir=ltr align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
Preparing and Loading Real Data
</font>
</h4>
<p dir=ltr style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
In this section, we load the real data used to train the model. This data includes input images (e.g., images of geometric shapes) and their corresponding masks, which specify the relevant class (background, circle, rectangle, triangle) for each pixel.
<br><b>Data Structure:</b>
<ul>
<li>Folder <code>output/train_images</code>: Contains input images in PNG format.</li>
<li>Folder <code>output/train_masks</code>: Contains labeled masks in PNG format where each pixel has a numeric class value.</li>
</ul>
In this cell, we first read the filenames of the images and masks, and then use a loop to load each image and its corresponding mask. If the mask is stored as RGB, we convert it to numeric labels using <code>argmax</code>. Finally, we convert the data into numpy arrays to prepare them for model training.
<br>We also display a few samples of the images and masks to verify the correctness of the loading process.
</font>
</p></div>

In [ ]:
images_dir = "/kaggle/input/datasets/rasoolestiri/u-net-dataset/train/images"
masks_dir = "/kaggle/input/datasets/rasoolestiri/u-net-dataset/train/masks"

image_files = sorted([f for f in os.listdir(images_dir) if f.endswith('.png')])
mask_files = sorted([f for f in os.listdir(masks_dir) if f.endswith('.png')])



In [ ]:
print(type(image_files[0]))

In [ ]:
IMG_SIZE = 128 
imgs = []
masks = []
    
for img_name, mask_name in zip(image_files, mask_files):
    img_path = os.path.join(images_dir, img_name)
    mask_path = os.path.join(masks_dir, mask_name)
    img = plt.imread(img_path)
    mask = plt.imread(mask_path)
    imgs.append(img)
    masks.append(mask)

In [ ]:
X = []
Y = []
for img, mask in zip(imgs, masks):
    if mask.ndim == 3:
        mask = np.argmax(mask, axis=-1)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)

    if img.ndim == 3:
        img = img[..., 0]
    img = img[..., np.newaxis]  # (128,128,1)

    X.append(img)
    Y.append(mask)

X = np.array(X).astype(np.float32)
Y = np.array(Y).astype(np.uint8)


In [ ]:
import random

random_index = []

while len(random_index) < 6:
    temp = random.randint(0, X.shape[0])
    if temp not in random_index:
        random_index.append(temp)

# نمایش چند نمونه تصویر و ماسک
for i in random_index:
    plt.figure(figsize=(6,3))
    plt.subplot(1,2,1)
    plt.imshow(X[i].squeeze(), cmap='gray')
    plt.title(('تصویر ورودی'))
    plt.axis('off')
    plt.subplot(1,2,2)
    plt.imshow(Y[i], cmap='tab10', vmin=0, vmax=3)
    plt.title(('ماسک (برچسب پیکسل‌ها)'))
    plt.axis('off')
    plt.show()

<h4 dir=ltr align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
Designing the UNet Model
</font>
</h4>
<p dir=ltr style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
In this section, we build the UNet model step-by-step without using pre-built functions. UNet is a well-known architecture for image segmentation that utilizes a downward path (encoder) for feature extraction and an upward path (decoder) for reconstructing the pixel label map. At each step of the decoder, corresponding features from the encoder are also added to the model (skip connection) to preserve spatial details. Finally, a convolution layer with the number of filters equal to the number of classes and a softmax activation function is placed to predict the probability of belonging to each class for every pixel.
</font>
</p>

In [ ]:
# تعیین تعداد کانال‌های ورودی بر اساس داده (برای تصاویر خاکستری = 1)
N_CHANNELS = 1  # اگر تصاویر ورودی شما تک‌کاناله (خاکستری) هستند
# تعداد کلاس‌ها (پس‌زمینه، دایره، مربع، مثلث)
NUM_CLASSES = 4

In [ ]:
# ورودی مدل (ابعاد تصویر و تعداد کانال)
inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, N_CHANNELS))

#Encoder 
c1 = layers.Conv2D(16, 3, padding='same', activation='relu')(inputs)
c1 = layers.Conv2D(16, 3, padding='same', activation='relu')(c1)
p1 = layers.MaxPooling2D((2,2))(c1)

c2 = layers.Conv2D(32, 3, padding='same', activation='relu')(p1)
c2 = layers.Conv2D(32, 3, padding='same', activation='relu')(c2)
p2 = layers.MaxPooling2D((2,2))(c2)

c3 = layers.Conv2D(64, 3, padding='same', activation='relu')(p2)
c3 = layers.Conv2D(64, 3, padding='same', activation='relu')(c3)
p3 = layers.MaxPooling2D((2,2))(c3)

c4 = layers.Conv2D(128, 3, padding='same', activation='relu')(p3)
c4 = layers.Conv2D(128, 3, padding='same', activation='relu')(c4)
p4 = layers.MaxPooling2D((2,2))(c4)

#Bottleneck Part 
bn = layers.Conv2D(256, 3, padding='same', activation='relu')(p4)
bn = layers.Conv2D(256, 3, padding='same', activation='relu')(bn)

# Decoder and Skip Connections 
u4 = layers.UpSampling2D((2,2))(bn)
u4 = layers.Concatenate()([u4, c4])
c5 = layers.Conv2D(128, 3, padding='same', activation='relu')(u4)
c5 = layers.Conv2D(128, 3, padding='same', activation='relu')(c5)

u3 = layers.UpSampling2D((2,2))(c5)
u3 = layers.Concatenate()([u3, c3])
c6 = layers.Conv2D(64, 3, padding='same', activation='relu')(u3)
c6 = layers.Conv2D(64, 3, padding='same', activation='relu')(c6)

u2 = layers.UpSampling2D((2,2))(c6)
u2 = layers.Concatenate()([u2, c2])
c7 = layers.Conv2D(32, 3, padding='same', activation='relu')(u2)
c7 = layers.Conv2D(32, 3, padding='same', activation='relu')(c7)

u1 = layers.UpSampling2D((2,2))(c7)
u1 = layers.Concatenate()([u1, c1])
c8 = layers.Conv2D(16, 3, padding='same', activation='relu')(u1)
c8 = layers.Conv2D(16, 3, padding='same', activation='relu')(c8)

# خروجی مدل (تعداد کلاس‌ها)
outputs = layers.Conv2D(NUM_CLASSES, 1, activation='softmax')(c8)

# ساخت مدل نهایی
model = models.Model(inputs, outputs)

model.summary()

## Why This U-Net Structure?

- **Encoder:** Extracts features and reduces spatial dimensions to better understand the image.

- **Bottleneck:** A compact, rich representation of the whole image.

- **Decoder + Skip Connections:** Reconstructs the image while preserving fine details.

- **Softmax at the output:** Each pixel must belong to exactly one class.

![](Unet.png)

<h4 dir=ltr align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
Splitting Data into Training and Validation Sets
</font>
</h4>
<p dir=ltr style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
To evaluate the model's performance, we split the data into two parts: <b>train</b> and <b>validation</b>. The training set is used for the model to learn, and the validation set is used to assess the quality of the model on data it has not seen before. This prevents overfitting (memorizing the data instead of true learning) and allows us to properly measure the model's performance.
<br>In this cell, using the <code>train_test_split</code> function, we split the data into an 80% training set and a 20% validation set.
</font>
</p>

In [ ]:
X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, random_state=42)

print(X_train.shape, Y_train.shape)
print(X_val.shape, Y_val.shape)

<h4 dir=ltr align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
Converting Masks to One-Hot Encoding
</font>
</h4>
<p dir=ltr style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
In a multi-class segmentation problem, each pixel must be assigned to one of several classes (background, circle, rectangle, triangle). To train the model, we need to convert the masks into <b>one-hot</b> format; meaning for each pixel, we create a vector with a length equal to the number of classes, where only the element corresponding to the correct class is 1 and the rest are 0. This allows the model to compare the softmax output with the ground truth and learn correctly.
<br>In this cell, this conversion is performed using <code>tf.one_hot</code>.
</font>
</p>

In [ ]:
# تبدیل ماسک‌های آموزش و اعتبارسنجی به فرمت one-hot
Y_train_oh = tf.one_hot(Y_train, depth=NUM_CLASSES)
Y_val_oh = tf.one_hot(Y_val, depth=NUM_CLASSES)

print(Y_train_oh.shape)
print(Y_val_oh.shape)

<h4 dir=ltr align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
Compiling the UNet Model
</font>
</h4>
<p dir=ltr style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
Before training the model, we need to <b>compile</b> it; meaning we specify the optimization algorithm (optimizer) for updating weights, the loss function (loss) for learning, and the metrics (metrics) for evaluating the model's performance.
<br>In this project:
<ul>
<li><b>optimizer:</b> The Adam algorithm for weight optimization</li>
<li><b>loss:</b> The <code>CategoricalCrossentropy</code> function to compare the softmax output with the one-hot mask</li>
<li><b>metrics:</b> The <code>MeanIoU</code> metric to measure segmentation accuracy</li>
</ul>
</font>
</p>

In [ ]:
import tensorflow as tf

class SparseMeanIoU(tf.keras.metrics.MeanIoU):
    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred = tf.argmax(y_pred, axis=-1)
        return super().update_state(y_true, y_pred, sample_weight)

In [ ]:
# Y_train شکل (N,128,128) با مقدارهای 0,1,2  ← one_hot نمی‌خواهد
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=[SparseMeanIoU(num_classes=NUM_CLASSES)]
)
model.fit(X_train, Y_train, validation_data=(X_val, Y_val), epochs=30)

<h4 dir=ltr align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
Training the UNet Model
</font>
</h4>
<p dir=ltr style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
In this stage, we train the model using the training data. In each <b>epoch</b> (training round), the model sees all the training data and improves its weights. After each epoch, the model is also evaluated on the validation data to ensure that it is not just memorizing the data and is actually learning.
<br>In this cell, we train the model using the <code>fit</code> method and save the loss history.
</font>
</p>

<h4 dir=ltr align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
Saving the Trained Model
</font>
</h4>
<p dir=ltr style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
After training is complete, we save the trained model so that we can later use it for prediction or continue training without needing to retrain from scratch. This is done using the <code>save</code> method, and the model is saved as an h5 file.
</font>
</p>

In [ ]:
model.save('unet_trained_model.keras')

In [ ]:
model = tf.keras.models.load_model(
    'unet_trained_model.keras',
    custom_objects={'SparseMeanIoU': SparseMeanIoU}
)

<h4 dir=ltr align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
Plotting the Loss Curve
</font>
</h4>
<p dir=ltr style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
To examine the model's learning progress, we plot the training and validation loss curves. This chart shows whether the model is learning properly or suffering from overfitting/underfitting. A decrease in loss across both sets indicates good model learning.
</font>
</p>

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
n_show = 5
for i in range(n_show):
    idx = np.random.randint(0, X_val.shape[0])
    img = X_val[idx]
    true_mask = Y_val[idx]
    pred_mask = model.predict(img[np.newaxis, ...])
    pred_mask = np.argmax(pred_mask[0], axis=-1)
    plt.figure(figsize=(10,3))
    plt.subplot(1,3,1)
    plt.imshow(img, cmap='gray')
    plt.axis('off')
    plt.subplot(1,3,2)
    plt.imshow(true_mask, cmap='tab10', vmin=0, vmax=3)
    plt.axis('off')
    plt.subplot(1,3,3)
    plt.imshow(pred_mask, cmap='tab10', vmin=0, vmax=3)
    plt.axis('off')
    plt.show()